# Experimento Mestre TCC (Kaggle & Local): Árvores Multimodais (`C1`/`C2`) vs. Full Fine-Tuning de Encoders Long-Context (`moBERTo`/`mmBERT`/`BERTimbau`) e Decoders (`Qwen 2.5`/`Bode 3.1`)

**Trabalho de Conclusão de Curso (TCC)**: *Detecção Precoce de Riscos e Atribuição Causal em Obras Públicas Brasileiras (Painel TCU)*  
**Versão do Pipeline**: `v2.0` (Auditado contra Sprints 1 a 8, Constituição SpecKit e `RELATORIO_TECNICO_FINAL_TCC.md`)  

---

## Sumário das Garantias Metodológicas e Correções da Auditoria (`v2.0`)
1. **Prevenção Absoluta de *Target Leakage* (`AUD-01`)**: Todas as colunas administrativas pós-fato (`situacao_cadastral`, `causa_cadastral`, `origem_rotulo`, `status_rotulagem`) e proxies operacionais (`has_embedding`, `num_docs_validos`, `num_chunks_validos`) foram expurgadas dos arquivos Parquet e são verificadas por asserção no carregamento.
2. **Alinhamento Exato com o Campeão da Sprint 8 (`AUD-02`)**: Os arquivos Parquet embutem na coluna `gte_c2` os vetores densos oficiais de 768 dimensões (`Alibaba-NLP/gte-multilingual-base`) extraídos dos **46.299 chunks autênticos em 3.827 obras** (pós-purga de 100% das fichas sintéticas FPDF), reproduzindo com fidelidade o benchmark campeão (`Macro-F1 = 0,5377`, `MCC = 0,5508`).
3. **Serialização Multimodal Segura (`AUD-03`)**: O cabeçalho opcional de metadados utiliza exclusivamente as 6 variáveis auditadas do perfil `c1_sem_proxies` (`base`, `uf`, `valor_investimento`, `valor_desbloqueado`, `exec_fisica`, `exec_financeira`), **sem injetar `id_tcu`** no texto para impedir memorização espúria de identificadores.
4. **Suficiência da Janela de Contexto e Checkpoints Nativos em Português (`AUD-04`)**:
   - No `BERTimbau` (`512` tokens), **97,2% das obras sofrem truncamento** e **66,3% dos tokens são descartados**.
   - Nos checkpoints **ModernBERT (8.192 tokens) nativos em português e multilíngues** (`Tropic-AI/moBERTo`, `Itau-Unibanco/NorBERTo-base` e `jhu-clsp/mmBERT-base`), o comprimento máximo real do corpus é de **2.270 tokens** (`p50 = 1.481`, `p90 = 1.743`), de modo que `max_length = 2048` cobre **99,95%** das obras sem corte e `4096` cobre **100,00%**!
5. **Portabilidade Local <-> Kaggle (`AUD-05`)**: Este notebook detecta automaticamente se está rodando localmente (VS Code / WSL) ou no Kaggle Kernels (`/kaggle/input`), ajustando caminhos e perfis de execução sem necessidade de editar o código.

In [1]:
# ==============================================================================
# SEÇÃO 0.1 — SETUP GLOBAL, DETECÇÃO DE AMBIENTE (LOCAL VS. KAGGLE) E SEEDS
# ==============================================================================
import gc
import json
import math
import os
import random
import time
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42

def fixar_seeds(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

fixar_seeds(SEED)

# Detecção automática de ambiente (Kaggle Kernels vs. Ambiente Local VS Code / WSL)
IS_KAGGLE = Path("/kaggle/input").exists() and not Path("c:/Users").exists()

# Modo de execução:
# - "FULL_KAGGLE": Executa 5 dobras completas para todos os Encoders (Full FT) e Decoders (QLoRA) + Bootstrap B=1000
# - "LOCAL_VALIDATION": Executa 100% da auditoria, 100% das Árvores (5 dobras OOF + Teste Retido),
#                       validação funcional end-to-end dos Encoders na GPU local e Bootstrap estatístico.
EXECUTION_MODE = os.environ.get(
    "TCC_EXECUTION_MODE",
    "FULL_KAGGLE" if IS_KAGGLE else "LOCAL_VALIDATION",
)

def localizar_diretorio_dados() -> Path:
    if IS_KAGGLE:
        candidatos = list(Path("/kaggle/input").rglob("treino_dev_completo.parquet"))
        if candidatos:
            return candidatos[0].parent
    for caminho in [
        Path("kaggle/datasets_preparados"),
        Path("./datasets_preparados"),
        Path("../kaggle/datasets_preparados"),
    ]:
        if (caminho / "treino_dev_completo.parquet").exists():
            return caminho.resolve()
    raise FileNotFoundError("Não foi possível localizar 'treino_dev_completo.parquet'.")

DATA_DIR = localizar_diretorio_dados()
OUTPUT_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("kaggle/resultados_validacao_local").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
BF16_SUPPORTED = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())

print("=" * 80)
print(f"AMBIENTE DETECTADO : {'KAGGLE KERNELS' if IS_KAGGLE else 'LOCAL (VS CODE / WORKSPACE)'}")
print(f"MODO DE EXECUÇÃO   : {EXECUTION_MODE}")
print(f"DIRETÓRIO DE DADOS : {DATA_DIR}")
print(f"DIRETÓRIO DE SAÍDA : {OUTPUT_DIR}")
print(f"DISPOSITIVO PYTORCH: {DEVICE} (GPUs disponíveis: {NUM_GPUS})")
if NUM_GPUS > 0:
    for idx in range(NUM_GPUS):
        props = torch.cuda.get_device_properties(idx)
        print(f"  -> GPU {idx}: {props.name} | VRAM Total: {props.total_memory / (1024**3):.2f} GB | BF16: {BF16_SUPPORTED}")
print("=" * 80)


AMBIENTE DETECTADO : LOCAL (VS CODE / WORKSPACE)
MODO DE EXECUÇÃO   : LOCAL_VALIDATION
DIRETÓRIO DE DADOS : C:\Users\lucas\OneDrive\Documentos\TCC\kaggle\datasets_preparados
DIRETÓRIO DE SAÍDA : C:\Users\lucas\OneDrive\Documentos\TCC\resultados_validacao_local
DISPOSITIVO PYTORCH: cuda (GPUs disponíveis: 1)
  -> GPU 0: NVIDIA GeForce RTX 5060 | VRAM Total: 7.93 GB | BF16: True


In [2]:
# ==============================================================================
# SEÇÃO 0.2 — CARREGAMENTO AUDITADO DOS PARQUETS E VERIFICAÇÃO ZERO-LEAKAGE
# ==============================================================================
NOMES_CLASSES = {
    0: "0_Regular",
    1: "1_Tecnica",
    2: "2_Orcamentaria",
    3: "3_Abandono",
}

NUMERIC_COLS_C1 = [
    "valor_investimento",
    "valor_desbloqueado",
    "exec_fisica",
    "exec_financeira",
]
CATEGORICAL_COLS_C1 = ["base", "uf"]
COLUNAS_TABULARES_C1 = NUMERIC_COLS_C1 + ["base_cat", "uf_cat"]

COLUNAS_PROIBIDAS_LEAKAGE = {
    "situacao_cadastral",
    "situacao_origem",
    "causa_cadastral",
    "origem_rotulo",
    "status_rotulagem",
    "has_embedding",
    "num_docs_validos",
    "num_chunks_validos",
    "embedding",
}

def extrair_features_tabulares_c1(
    df: pd.DataFrame,
    categorical_mappings: Optional[Dict[str, Dict[str, int]]] = None,
    numeric_imputations: Optional[Dict[str, float]] = None,
) -> Tuple[np.ndarray, Dict[str, Dict[str, int]], Dict[str, float]]:
    """Extrai os 6 atributos do perfil oficial `c1_sem_proxies` sem vazamento entre treino e teste.
    Reproduz exatamente `src.modeling.loader.extract_tabular_features(feature_profile='c1_sem_proxies')`.
    """
    df_feat = pd.DataFrame(index=df.index)
    new_imputations: Dict[str, float] = {}

    for col in NUMERIC_COLS_C1:
        s = pd.to_numeric(df[col], errors="coerce")
        if col in ("exec_fisica", "exec_financeira"):
            df_feat[col] = s.fillna(0.0).astype(np.float32)
        else:
            if numeric_imputations and col in numeric_imputations:
                med = numeric_imputations[col]
            else:
                med = float(s.dropna().median()) if s.dropna().size > 0 else 0.0
                new_imputations[col] = med
            df_feat[col] = s.fillna(med).astype(np.float32)

    new_mappings: Dict[str, Dict[str, int]] = {}
    for col in CATEGORICAL_COLS_C1:
        s_cat = df[col].astype(str).fillna("DESCONHECIDO")
        if categorical_mappings and col in categorical_mappings:
            cat_map = categorical_mappings[col]
        else:
            uniques = sorted(s_cat.unique().tolist())
            cat_map = {val: idx for idx, val in enumerate(uniques)}
            new_mappings[col] = cat_map
        df_feat[f"{col}_cat"] = s_cat.map(cat_map).fillna(-1).astype(np.float32)

    X_tab = df_feat[COLUNAS_TABULARES_C1].to_numpy(dtype=np.float32)
    maps = new_mappings or (categorical_mappings or {})
    imps = new_imputations or (numeric_imputations or {})
    return X_tab, maps, imps

df_dev = pd.read_parquet(DATA_DIR / "treino_dev_completo.parquet")
df_test = pd.read_parquet(DATA_DIR / "teste_retido_completo.parquet")

# 1. Auditoria contra colunas proibidas (Target Leakage & Proxies Operacionais)
vazadas_dev = COLUNAS_PROIBIDAS_LEAKAGE.intersection(df_dev.columns)
vazadas_test = COLUNAS_PROIBIDAS_LEAKAGE.intersection(df_test.columns)
assert not vazadas_dev and not vazadas_test, f"ERRO CRÍTICO DE LEAKAGE: {vazadas_dev | vazadas_test}"

# 2. Auditoria de disjunção estrita de grupos (id_tcu) entre Treino/Dev e Teste Retido
ids_dev = set(df_dev["id_tcu"].astype(str))
ids_test = set(df_test["id_tcu"].astype(str))
intersecao_dev_test = ids_dev.intersection(ids_test)
assert len(intersecao_dev_test) == 0, f"ERRO: {len(intersecao_dev_test)} id_tcu vazados entre treino_dev e teste_retido!"

# 3. Auditoria de disjunção de id_tcu entre as 5 dobras de validação cruzada
folds_unicos = sorted(df_dev["fold_cv"].unique().tolist())
assert folds_unicos == [1, 2, 3, 4, 5], f"Dobras inesperadas: {folds_unicos}"
for f1 in folds_unicos:
    ids_f1 = set(df_dev.loc[df_dev["fold_cv"] == f1, "id_tcu"])
    for f2 in folds_unicos:
        if f1 < f2:
            ids_f2 = set(df_dev.loc[df_dev["fold_cv"] == f2, "id_tcu"])
            assert len(ids_f1.intersection(ids_f2)) == 0, f"Vazamento de id_tcu entre dobra {f1} e {f2}!"

# 4. Construção das matrizes C1 (Tabular Limpo 6d) e C2 (Multimodal 774d = 6d + 768d GTE)
X_tab_dev, cat_maps_dev, num_imps_dev = extrair_features_tabulares_c1(df_dev)
X_tab_test, _, _ = extrair_features_tabulares_c1(df_test, categorical_mappings=cat_maps_dev, numeric_imputations=num_imps_dev)

X_emb_dev = np.vstack(df_dev["gte_c2"].to_numpy()).astype(np.float32)
X_emb_test = np.vstack(df_test["gte_c2"].to_numpy()).astype(np.float32)
assert X_emb_dev.shape == (len(df_dev), 768) and X_emb_test.shape == (len(df_test), 768)

X_comb_dev = np.hstack([X_tab_dev, X_emb_dev]).astype(np.float32)
X_comb_test = np.hstack([X_tab_test, X_emb_test]).astype(np.float32)

y_dev = df_dev["label_risco"].to_numpy(dtype=np.int64)
y_test = df_test["label_risco"].to_numpy(dtype=np.int64)
folds_dev = df_dev["fold_cv"].to_numpy(dtype=np.int64)

mask_texto_dev = df_dev["tem_texto"].to_numpy(dtype=bool)
mask_texto_test = df_test["tem_texto"].to_numpy(dtype=bool)

print("[AUDITORIA ZERO-LEAKAGE APROVADA COM SUCESSO]")
print(f"  -> treino_dev : {len(df_dev):,d} obras totais | {int(mask_texto_dev.sum()):,d} obras com texto autêntico ({mask_texto_dev.mean()*100:.1f}%)")
print(f"  -> teste_retido: {len(df_test):,d} obras totais | {int(mask_texto_test.sum()):,d} obras com texto autêntico ({mask_texto_test.mean()*100:.1f}%)")
print(f"  -> Dimensões  : X_tab = {X_tab_dev.shape[1]}d | X_emb (gte_c2) = {X_emb_dev.shape[1]}d | X_comb (C2) = {X_comb_dev.shape[1]}d")

# Tabela de distribuição de classes
linhas_dist = []
for c, nome in NOMES_CLASSES.items():
    linhas_dist.append({
        "Classe": nome,
        "Treino/Dev Total (14.522)": int((y_dev == c).sum()),
        "Treino/Dev Com Texto (3.088)": int((y_dev[mask_texto_dev] == c).sum()),
        "Teste Retido Total (3.632)": int((y_test == c).sum()),
        "Teste Retido Com Texto (739)": int((y_test[mask_texto_test] == c).sum()),
    })
df_distribuicao = pd.DataFrame(linhas_dist)
display(df_distribuicao)


[AUDITORIA ZERO-LEAKAGE APROVADA COM SUCESSO]
  -> treino_dev : 14,522 obras totais | 3,088 obras com texto autêntico (21.3%)
  -> teste_retido: 3,632 obras totais | 739 obras com texto autêntico (20.3%)
  -> Dimensões  : X_tab = 6d | X_emb (gte_c2) = 768d | X_comb (C2) = 774d


,Classe,Treino/Dev Total (14.522),Treino/Dev Com Texto (3.088),Teste Retido Total (3.632),Teste Retido Com Texto (739)
0,0_Regular,13319,2359,3330,562
1,1_Tecnica,782,672,196,163
2,2_Orcamentaria,83,10,21,2
3,3_Abandono,338,47,85,12


---
## Seção 1 — Diagnóstico Empírico da Janela de Contexto (512 vs. 2.048 / 4.096 / 8.192 Tokens)

Na Sprint 8 local, o modelo `BERTimbau + LoRA` (`neuralmind/bert-base-portuguese-cased`) atingiu apenas **Macro-F1 = 0,3269** no teste retido, sendo amplamente superado pelo `Hierarquico_C2_LightGBM` (**Macro-F1 = 0,5377**).

Abaixo demonstramos empiricamente a causa estrutural desse resultado:
1. O `BERTimbau` possui `max_position_embeddings = 512`, truncando **97,2% das obras** e descartando **66,3% de todos os subword tokens** dos relatórios técnicos e termos aditivos recuperados pelo RAG.
2. Já os encoders de arquitetura **ModernBERT** com suporte ao português (`Tropic-AI/moBERTo`, `Itau-Unibanco/NorBERTo-base` e `jhu-clsp/mmBERT-base`) possuem janela nativa de **8.192 tokens** via *Rotary Position Embeddings (RoPE)* e *Unpadding*. Como o maior dossiê textual do nosso corpus possui **2.270 tokens** (`p90 = 1.743 tokens`), configurar `max_length = 2048` (ou `2560`) processa **praticamente 100% do texto integral sem truncamento** e com consumo enxuto de VRAM (~5,8 GB em FP16).

In [3]:
# ==============================================================================
# SEÇÃO 1.1 — MEDIÇÃO EMPÍRICA DE SUBWORD TOKENS E SERIALIZAÇÃO SEGURA DE TEXTO
# ==============================================================================
from transformers import AutoTokenizer

def formatar_entrada_llm_segura(row: pd.Series, incluir_cabecalho_tabular: bool = False) -> str:
    """Formata o texto da obra opcionalmente precedido pelas 6 variáveis tabulares limpas (C1).
    NUNCA inclui `id_tcu` ou variáveis administrativas pós-fato para evitar vazamento/memorização.
    """
    texto_doc = str(row.get("texto_integral", "") or "").strip()
    if not incluir_cabecalho_tabular:
        return texto_doc
    cabecalho = (
        f"[METADADOS C1 AUDITADOS] "
        f"FONTE: {row.get('base', 'N/D')} | UF: {row.get('uf', 'N/D')} | "
        f"INVESTIMENTO: R$ {float(row.get('valor_investimento', 0.0)):,.2f} | "
        f"DESBLOQUEADO: R$ {float(row.get('valor_desbloqueado', 0.0)):,.2f} | "
        f"EXEC_FISICA: {float(row.get('exec_fisica', 0.0)):.1f}% | "
        f"EXEC_FINANCEIRA: {float(row.get('exec_financeira', 0.0)):.1f}%\n"
        f"[DOSSIÊ DOCUMENTAL AUTÊNTICO]\n"
    )
    return cabecalho + texto_doc

textos_dev_autenticos = df_dev.loc[mask_texto_dev, "texto_integral"].tolist()
textos_test_autenticos = df_test.loc[mask_texto_test, "texto_integral"].tolist()

CHECKPOINTS_DIAGNOSTICO = [
    ("BERTimbau-base (512)", "neuralmind/bert-base-portuguese-cased", 512),
    ("moBERTo PT-BR (ModernBERT 8192)", "Tropic-AI/moBERTo", 8192),
]

relatorio_tokens = []
for rotulo, ckpt, max_pos_nativo in CHECKPOINTS_DIAGNOSTICO:
    tok = AutoTokenizer.from_pretrained(ckpt)
    enc = tok(textos_dev_autenticos, truncation=False, add_special_tokens=True)
    lens = np.array([len(ids) for ids in enc["input_ids"]], dtype=np.int32)
    perda_512 = np.maximum(0, lens - 512).sum() / max(1, lens.sum())
    perda_2048 = np.maximum(0, lens - 2048).sum() / max(1, lens.sum())
    relatorio_tokens.append({
        "Modelo / Tokenizador": rotulo,
        "Checkpoint HF": ckpt,
        "Janela Nativa": max_pos_nativo,
        "Mediana (p50)": int(np.percentile(lens, 50)),
        "p75": int(np.percentile(lens, 75)),
        "p90": int(np.percentile(lens, 90)),
        "Máximo": int(lens.max()),
        "% Obras > 512": round(float((lens > 512).mean() * 100), 2),
        "% Tokens Perdidos em 512": round(float(perda_512 * 100), 2),
        "% Obras > 2048": round(float((lens > 2048).mean() * 100), 2),
        "% Tokens Perdidos em 2048": round(float(perda_2048 * 100), 3),
    })

df_diagnostico_tokens = pd.DataFrame(relatorio_tokens)
print("[DIAGNÓSTICO EMPÍRICO DE JANELA DE CONTEXTO NAS 3.088 OBRAS DE TREINO COM TEXTO]")
display(df_diagnostico_tokens)


[DIAGNÓSTICO EMPÍRICO DE JANELA DE CONTEXTO NAS 3.088 OBRAS DE TREINO COM TEXTO]


,Modelo / Tokenizador,Checkpoint HF,Janela Nativa,Mediana (p50),p75,p90,Máximo,% Obras > 512,% Tokens Perdidos em 512,% Obras > 2048,% Tokens Perdidos em 2048
0,BERTimbau-base (512),neuralmind/bert-base-portuguese-cased,512,1594,1794,1802,1805,97.22,66.31,0.0,0.000
1,moBERTo PT-BR (ModernBERT 8192),Tropic-AI/moBERTo,8192,1481,1660,1743,2270,96.02,64.32,0.1,0.007


---
## Seção 2 — Família 1: Árvores de Decisão & Ensembles Multimodais (`C1` Tabular vs. `C2` Multimodal GTE 768d)

Nesta seção, treinamos e avaliamos sistematicamente todos os algoritmos baseados em árvores de decisão sob validação cruzada de 5 dobras agrupadas por `id_tcu` (`oof_proba`) e no teste retido congelado ($N = 3.632$ obras totais e $N_{\text{texto}} = 739$ obras com texto autêntico):

1. **`Dummy_Majority_C1`**: Piso inferior (*Zero-R*).
2. **`RandomForest` (`C1` Flat e `C2` Flat)**: *Bagging* de árvores com `class_weight="balanced_subsample"`.
3. **`LightGBM` (`C1` Flat, `C2` Flat, `C1` Hierárquico e `C2` Hierárquico)**: Motor baseado em histogramas e GOSS, incluindo a reprodução exata do campeão da Sprint 8 (`TwoStageHierarchicalClassifier` de `src/modeling/hierarchical.py`).
4. **`XGBoost` (`C1` Flat, `C2` Flat e `C2` Hierárquico)**: *Gradient Boosting* com `tree_method="hist"` e regularização $L_1/L_2$.
5. **`CatBoost` (`C1` Flat, `C2` Flat e `C2` Hierárquico)**: *Oblivious Trees* simétricas com ponderação balanceada de classes.

In [4]:
# ==============================================================================
# SEÇÃO 2.1 — FUNÇÕES DE MÉTRICAS OFICIAIS E ARQUITETURA HIERÁRQUICA EM 2 ESTÁGIOS
# ==============================================================================
import lightgbm as lgb
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

def compute_balanced_sample_weights(y: np.ndarray) -> np.ndarray:
    classes, counts = np.unique(y, return_counts=True)
    total = len(y)
    n_classes = len(classes)
    weight_map = {int(c): total / (n_classes * int(cnt)) for c, cnt in zip(classes, counts)}
    return np.array([weight_map[int(val)] for val in y], dtype=np.float32)

def normalizar_probabilidades(proba: np.ndarray) -> np.ndarray:
    proba = np.clip(np.asarray(proba, dtype=np.float64), 1e-12, 1.0)
    somas = proba.sum(axis=1, keepdims=True)
    somas = np.where(somas <= 0, 1.0, somas)
    return (proba / somas).astype(np.float32)

def calcular_metricas_completas(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    y_pred: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    y_proba = normalizar_probabilidades(y_proba)
    if y_pred is None:
        y_pred = np.argmax(y_proba, axis=1)

    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    weighted_f1 = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    bal_acc = float(balanced_accuracy_score(y_true, y_pred))
    mcc = float(matthews_corrcoef(y_true, y_pred))
    acc = float(accuracy_score(y_true, y_pred))

    f1_per_class = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2, 3], zero_division=0)
    rec_per_class = recall_score(y_true, y_pred, average=None, labels=[0, 1, 2, 3], zero_division=0)
    prec_per_class = precision_score(y_true, y_pred, average=None, labels=[0, 1, 2, 3], zero_division=0)

    pr_auc_per_class = {}
    for c, nome in NOMES_CLASSES.items():
        y_bin_c = (y_true == c).astype(int)
        pr_auc_per_class[nome] = round(
            float(average_precision_score(y_bin_c, y_proba[:, c])) if y_bin_c.sum() > 0 else 0.0,
            4,
        )

    # Métricas binárias de detecção de risco (Estágio 1: Regular=0 vs Risco={1,2,3})
    y_true_bin = (y_true > 0).astype(int)
    p_risco = 1.0 - y_proba[:, 0]
    y_pred_bin = (p_risco >= 0.5).astype(int)
    sens_risco = float(recall_score(y_true_bin, y_pred_bin, pos_label=1, zero_division=0))
    espec_reg = float(recall_score(y_true_bin, y_pred_bin, pos_label=0, zero_division=0))
    bal_acc_bin = 0.5 * (sens_risco + espec_reg)
    f1_bin_macro = float(f1_score(y_true_bin, y_pred_bin, average="macro", zero_division=0))

    return {
        "macro_f1": round(macro_f1, 4),
        "balanced_accuracy": round(bal_acc, 4),
        "mcc": round(mcc, 4),
        "weighted_f1": round(weighted_f1, 4),
        "accuracy": round(acc, 4),
        "bin_macro_f1": round(f1_bin_macro, 4),
        "bin_sensibilidade_risco": round(sens_risco, 4),
        "bin_especificidade_regular": round(espec_reg, 4),
        "bin_balanced_accuracy": round(bal_acc_bin, 4),
        "f1_por_classe": {NOMES_CLASSES[i]: round(float(f1_per_class[i]), 4) for i in range(4)},
        "recall_por_classe": {NOMES_CLASSES[i]: round(float(rec_per_class[i]), 4) for i in range(4)},
        "precision_por_classe": {NOMES_CLASSES[i]: round(float(prec_per_class[i]), 4) for i in range(4)},
        "pr_auc_por_classe": pr_auc_per_class,
        "matriz_confusao": confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3]).tolist(),
    }


def instanciar_backbone_arvore(backbone: str, n_classes: int, seed: int = SEED):
    use_gpu = torch.cuda.is_available()
    if backbone == "lightgbm":
        params = dict(
            n_estimators=120 if n_classes == 2 else 100,
            learning_rate=0.05,
            num_leaves=35 if n_classes == 2 else 31,
            max_depth=6,
            min_child_samples=20 if n_classes == 2 else 10,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1,
            verbose=-1,
        )
        if n_classes == 2:
            return LGBMClassifier(objective="binary", **params)
        return LGBMClassifier(objective="multiclass", num_class=n_classes, **params)

    if backbone == "xgboost":
        params = dict(
            n_estimators=120,
            learning_rate=0.05,
            max_depth=6,
            max_bin=64,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            tree_method="hist",
            device="cuda" if use_gpu else "cpu",
            random_state=seed,
            n_jobs=-1,
            verbosity=0,
        )
        if n_classes == 2:
            return XGBClassifier(objective="binary:logistic", eval_metric="logloss", **params)
        return XGBClassifier(objective="multi:softprob", num_class=n_classes, eval_metric="mlogloss", **params)

    if backbone == "catboost":
        loss = "Logloss" if n_classes == 2 else "MultiClass"
        return CatBoostClassifier(
            iterations=200,
            learning_rate=0.06,
            depth=6,
            border_count=64,
            loss_function=loss,
            auto_class_weights="SqrtBalanced",
            task_type="GPU" if use_gpu else "CPU",
            allow_writing_files=False,
            random_seed=seed,
            verbose=False,
        )

    if backbone == "random_forest":
        return RandomForestClassifier(
            n_estimators=150,
            max_depth=12,
            min_samples_leaf=3,
            class_weight="balanced_subsample",
            random_state=seed,
            n_jobs=-1,
        )

    raise ValueError(f"Backbone desconhecido: {backbone}")


def treinar_predizer_arvore_flat(backbone: str, X_tr: np.ndarray, y_tr: np.ndarray, X_ev: np.ndarray, seed: int = SEED) -> np.ndarray:
    if backbone == "dummy":
        clf = DummyClassifier(strategy="prior", random_state=seed)
        clf.fit(X_tr, y_tr)
        return clf.predict_proba(X_ev).astype(np.float32)

    clf = instanciar_backbone_arvore(backbone, n_classes=4, seed=seed)
    if backbone == "xgboost":
        sw = compute_balanced_sample_weights(y_tr)
        clf.fit(X_tr, y_tr, sample_weight=sw)
    else:
        clf.fit(X_tr, y_tr)
    return normalizar_probabilidades(clf.predict_proba(X_ev))


class TwoStageHierarchicalClassifier:
    """Classificador Hierárquico em 2 Estágios compatível com LightGBM (idêntico a `src.modeling.hierarchical`), XGBoost e CatBoost.
    - Estágio 1: Classificador Binário (0 = Regular vs. 1 = Risco {1, 2, 3})
    - Estágio 2: Classificador Multiclasse Condicional nas 3 causas de risco ({1, 2, 3} -> {0, 1, 2})
    - Combinação Probabilística: P(Y=0) = 1 - P_risco; P(Y=c) = P_risco * P(Y=c | Y>0)
    """
    def __init__(self, backbone: str = "lightgbm", seed: int = SEED):
        self.backbone = backbone
        self.seed = seed
        self.stage1 = None
        self.stage2 = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "TwoStageHierarchicalClassifier":
        y_bin = (y > 0).astype(np.int32)
        mask_risk = y > 0
        X_risk = X[mask_risk]
        y_risk = (y[mask_risk] - 1).astype(np.int32)

        if self.backbone == "lightgbm":
            # Reprodução exata do BinaryRiskDetector e ConditionalCausalClassifier da Sprint 7/8
            n_neg = int(np.sum(y_bin == 0))
            n_pos = max(1, int(np.sum(y_bin == 1)))
            scale_pos_weight = float(n_neg / n_pos)
            ds1 = lgb.Dataset(X, label=y_bin, free_raw_data=False)
            params1 = {
                "objective": "binary",
                "metric": "binary_logloss",
                "scale_pos_weight": scale_pos_weight,
                "learning_rate": 0.05,
                "num_leaves": 35,
                "max_depth": 6,
                "min_child_samples": 20,
                "feature_fraction": 0.8,
                "seed": self.seed,
                "verbose": -1,
                "num_threads": 4,
                "max_bin": 255,
            }
            self.stage1 = lgb.train(params1, ds1, num_boost_round=120)

            sw2 = compute_balanced_sample_weights(y_risk)
            ds2 = lgb.Dataset(X_risk, label=y_risk, weight=sw2, free_raw_data=False)
            params2 = {
                "objective": "multiclass",
                "num_class": 3,
                "learning_rate": 0.05,
                "num_leaves": 31,
                "max_depth": 6,
                "min_child_samples": 10,
                "feature_fraction": 0.8,
                "seed": self.seed,
                "verbose": -1,
                "num_threads": 4,
                "max_bin": 255,
            }
            self.stage2 = lgb.train(params2, ds2, num_boost_round=100)
            return self

        self.stage1 = instanciar_backbone_arvore(self.backbone, n_classes=2, seed=self.seed)
        self.stage2 = instanciar_backbone_arvore(self.backbone, n_classes=3, seed=self.seed + 100)

        if self.backbone == "xgboost":
            sw1 = compute_balanced_sample_weights(y_bin)
            self.stage1.fit(X, y_bin, sample_weight=sw1)
            sw2 = compute_balanced_sample_weights(y_risk)
            self.stage2.fit(X_risk, y_risk, sample_weight=sw2)
        else:
            self.stage1.fit(X, y_bin)
            self.stage2.fit(X_risk, y_risk)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        if self.backbone == "lightgbm":
            p_pos = np.asarray(self.stage1.predict(X), dtype=np.float32).reshape(-1, 1)
            p_reg = 1.0 - p_pos
            p_stage2 = np.asarray(self.stage2.predict(X), dtype=np.float32).reshape(len(X), 3)
            p_joint = np.hstack([p_reg, p_pos * p_stage2])
            return normalizar_probabilidades(p_joint)

        p_stage1 = self.stage1.predict_proba(X)
        p_reg = p_stage1[:, 0:1]
        p_risk = p_stage1[:, 1:2]
        p_stage2 = self.stage2.predict_proba(X)
        p_joint = np.hstack([p_reg, p_risk * p_stage2])
        return normalizar_probabilidades(p_joint)


In [5]:
# ==============================================================================
# SEÇÃO 2.2 — TREINAMENTO 5-FOLD OOF E AVALIAÇÃO NO TESTE RETIDO (FAMÍLIA ÁRVORES)
# ==============================================================================
CATALOGO_ARVORES = [
    ("Dummy_Majority_C1", "dummy", "C1", False),
    ("RandomForest_C1_Tabular", "random_forest", "C1", False),
    ("RandomForest_C2_Multimodal", "random_forest", "C2", False),
    ("LightGBM_C1_Tabular", "lightgbm", "C1", False),
    ("LightGBM_C2_Multimodal", "lightgbm", "C2", False),
    ("XGBoost_C1_Tabular", "xgboost", "C1", False),
    ("XGBoost_C2_Multimodal", "xgboost", "C2", False),
    ("CatBoost_C1_Tabular", "catboost", "C1", False),
    ("CatBoost_C2_Multimodal", "catboost", "C2", False),
    ("Hierarquico_C1_LightGBM", "lightgbm", "C1", True),
    ("Hierarquico_C2_LightGBM", "lightgbm", "C2", True),
    ("Hierarquico_C2_XGBoost", "xgboost", "C2", True),
    ("Hierarquico_C2_CatBoost", "catboost", "C2", True),
]

PREDICOES_OOF: Dict[str, np.ndarray] = {}
PREDICOES_TESTE_INTEGRAL: Dict[str, np.ndarray] = {}
PREDICOES_TESTE_TEXTO: Dict[str, np.ndarray] = {}
RESULTADOS_MODELOS: Dict[str, Dict[str, Any]] = {}
MODELOS_TREINADOS_ARVORE: Dict[str, Any] = {}

print("Iniciando avaliação exaustiva da Família 1 (Árvores de Decisão & Ensembles)...")
for nome_modelo, backbone, espaco_features, is_hierarquico in CATALOGO_ARVORES:
    t0 = time.perf_counter()
    X_dev_atual = X_tab_dev if espaco_features == "C1" else X_comb_dev
    X_test_atual = X_tab_test if espaco_features == "C1" else X_comb_test

    oof_proba = np.zeros((len(df_dev), 4), dtype=np.float32)
    for f in folds_unicos:
        tr_idx = folds_dev != f
        va_idx = folds_dev == f
        if is_hierarquico:
            clf_fold = TwoStageHierarchicalClassifier(backbone=backbone, seed=SEED)
            clf_fold.fit(X_dev_atual[tr_idx], y_dev[tr_idx])
            oof_proba[va_idx] = clf_fold.predict_proba(X_dev_atual[va_idx])
        else:
            oof_proba[va_idx] = treinar_predizer_arvore_flat(
                backbone, X_dev_atual[tr_idx], y_dev[tr_idx], X_dev_atual[va_idx], seed=SEED
            )

    # Ajuste final em 100% do treino_dev para avaliação única no teste_retido congelado
    if is_hierarquico:
        clf_final = TwoStageHierarchicalClassifier(backbone=backbone, seed=SEED)
        clf_final.fit(X_dev_atual, y_dev)
        test_proba = clf_final.predict_proba(X_test_atual)
        MODELOS_TREINADOS_ARVORE[nome_modelo] = clf_final
    else:
        test_proba = treinar_predizer_arvore_flat(backbone, X_dev_atual, y_dev, X_test_atual, seed=SEED)

    dt = time.perf_counter() - t0
    PREDICOES_OOF[nome_modelo] = oof_proba
    PREDICOES_TESTE_INTEGRAL[nome_modelo] = test_proba
    PREDICOES_TESTE_TEXTO[nome_modelo] = test_proba[mask_texto_test]

    m_oof = calcular_metricas_completas(y_dev, oof_proba)
    m_test_all = calcular_metricas_completas(y_test, test_proba)
    m_test_txt = calcular_metricas_completas(y_test[mask_texto_test], test_proba[mask_texto_test])

    RESULTADOS_MODELOS[nome_modelo] = {
        "familia": "1_Arvore_Hierarquica" if is_hierarquico else "1_Arvore_Flat",
        "espaco_features": espaco_features,
        "tempo_segundos": round(dt, 2),
        "oof_integral": m_oof,
        "teste_integral_3632": m_test_all,
        "teste_com_texto_739": m_test_txt,
    }
    print(
        f"  [OK] {nome_modelo:26s} | OOF F1={m_oof['macro_f1']:.4f} | "
        f"Teste(3632) F1={m_test_all['macro_f1']:.4f} MCC={m_test_all['mcc']:.4f} | "
        f"TesteTexto(739) F1={m_test_txt['macro_f1']:.4f} ({dt:.1f}s)"
    )

# Exibir ranking das Árvores
linhas_arvores = []
for nome, res in RESULTADOS_MODELOS.items():
    t_all = res["teste_integral_3632"]
    t_txt = res["teste_com_texto_739"]
    linhas_arvores.append({
        "Modelo": nome,
        "Features": res["espaco_features"],
        "OOF Macro-F1": res["oof_integral"]["macro_f1"],
        "Teste(3632) Macro-F1": t_all["macro_f1"],
        "Teste(3632) MCC": t_all["mcc"],
        "Teste(3632) Sens. Risco": t_all["bin_sensibilidade_risco"],
        "TesteTexto(739) Macro-F1": t_txt["macro_f1"],
        "TesteTexto(739) MCC": t_txt["mcc"],
        "Tempo (s)": res["tempo_segundos"],
    })
df_ranking_arvores = pd.DataFrame(linhas_arvores).sort_values("Teste(3632) Macro-F1", ascending=False)
display(df_ranking_arvores)


Iniciando avaliação exaustiva da Família 1 (Árvores de Decisão & Ensembles)...
  [OK] Dummy_Majority_C1          | OOF F1=0.2392 | Teste(3632) F1=0.2392 MCC=0.0000 | TesteTexto(739) F1=0.2160 (0.0s)


  [OK] RandomForest_C1_Tabular    | OOF F1=0.4542 | Teste(3632) F1=0.4616 MCC=0.4029 | TesteTexto(739) F1=0.5972 (4.5s)


  [OK] RandomForest_C2_Multimodal | OOF F1=0.4909 | Teste(3632) F1=0.4775 MCC=0.4891 | TesteTexto(739) F1=0.5047 (7.3s)


  [OK] LightGBM_C1_Tabular        | OOF F1=0.4385 | Teste(3632) F1=0.4338 MCC=0.3953 | TesteTexto(739) F1=0.5987 (1.8s)


  [OK] LightGBM_C2_Multimodal     | OOF F1=0.5155 | Teste(3632) F1=0.5166 MCC=0.5258 | TesteTexto(739) F1=0.6119 (89.7s)


...

  [OK] Hierarquico_C1_LightGBM    | OOF F1=0.4588 | Teste(3632) F1=0.4462 MCC=0.3965 | TesteTexto(739) F1=0.5950 (0.8s)


  [OK] Hierarquico_C2_LightGBM    | OOF F1=0.5384 | Teste(3632) F1=0.5377 MCC=0.5508 | TesteTexto(739) F1=0.5810 (34.1s)


  [OK] Hierarquico_C2_XGBoost     | OOF F1=0.5424 | Teste(3632) F1=0.5361 MCC=0.5510 | TesteTexto(739) F1=0.5888 (8.5s)


  [OK] Hierarquico_C2_CatBoost    | OOF F1=0.5545 | Teste(3632) F1=0.5322 MCC=0.5568 | TesteTexto(739) F1=0.6097 (28.4s)


,Modelo,Features,OOF Macro-F1,Teste(3632) Macro-F1,Teste(3632) MCC,Teste(3632) Sens. Risco,TesteTexto(739) Macro-F1,TesteTexto(739) MCC,Tempo (s)
8,CatBoost_C2_Multimodal,C2,0.5431,0.5932,0.5817,0.8510,0.6137,0.6162,17.03
10,Hierarquico_C2_LightGBM,C2,0.5384,0.5377,0.5508,0.9040,0.5810,0.5905,34.06
11,Hierarquico_C2_XGBoost,C2,0.5424,0.5361,0.5510,0.9040,0.5888,0.5922,8.55
12,Hierarquico_C2_CatBoost,C2,0.5545,0.5322,0.5568,0.7219,0.6097,0.5693,28.44
4,LightGBM_C2_Multimodal,C2,0.5155,0.5166,0.5258,0.9272,0.6119,0.6036,89.71
6,XGBoost_C2_Multimodal,C2,0.5146,0.5121,0.5233,0.9503,0.6089,0.6120,9.67
7,CatBoost_C1_Tabular,C1,0.4868,0.5054,0.4193,0.7185,0.5982,0.4713,10.17
2,RandomForest_C2_Multimodal,C2,0.4909,0.4775,0.4891,0.9106,0.5047,0.4801,7.32
1,RandomForest_C1_Tabular,C1,0.4542,0.4616,0.4029,0.8675,0.5972,0.5797,4.46
9,Hierarquico_C1_LightGBM,C1,0.4588,0.4462,0.3965,0.8940,0.5950,0.5842,0.85


---
## Seção 3 — Família 2: *Full Fine-Tuning* de Encoders Bidirecionais (Curto vs. Longo Contexto em Português)

Para testar diretamente a Hipótese Científica $H_1$ (*o limite rígido de 512 tokens e o ajuste raso via LoRA foram os gargalos do BERTimbau na Sprint 8*), executamos o **Full Fine-Tuning (100% dos parâmetros descongelados)** comparando:

1. **`BERTimbau_FullFT_512` (`neuralmind/bert-base-portuguese-cased`, `max_length = 512`)**:
   - Isola o efeito de descongelar 100% dos 109M parâmetros mantendo a janela curta de 512 tokens.
2. **`moBERTo_ModernBERT_PT_2048` (`Tropic-AI/moBERTo`, `max_length = 2048`)**:
   - Arquitetura **ModernBERT** pré-treinada em português com *Rotary Position Embeddings (RoPE)* e janela nativa de 8.192 tokens. Em `max_length = 2048`, **99,95% das obras do corpus são lidas integralmente sem nenhum truncamento**.
3. **`mmBERT_ModernBERT_Multi_2048` (`jhu-clsp/mmBERT-base`, `max_length = 2048`)**:
   - Arquitetura **ModernBERT Multilíngue** (149M parâmetros, janela de 8.192 tokens), servindo como segundo encoder de longo contexto para validação cruzada.

In [6]:
# ==============================================================================
# SEÇÃO 3.1 — MOTOR PYTORCH DE FULL FINE-TUNING PARA ENCODERS (512 VS. 2048+ TOKENS)
# ==============================================================================
from transformers import AutoConfig, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup

class ObrasDocumentDataset(Dataset):
    def __init__(self, textos: List[str], labels: np.ndarray):
        self.textos = textos
        self.labels = np.asarray(labels, dtype=np.int64)

    def __len__(self) -> int:
        return len(self.textos)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return {"texto": self.textos[idx], "label": int(self.labels[idx])}


def criar_collate_fn(tokenizer, max_length: int):
    def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        textos = [item["texto"] for item in batch]
        labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
        enc = tokenizer(
            textos,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        enc["labels"] = labels
        return enc
    return collate_fn


def calcular_pesos_classe_tensor(y_fold_train: np.ndarray, num_labels: int, device: torch.device) -> torch.Tensor:
    counts = np.bincount(y_fold_train, minlength=num_labels).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    total = float(len(y_fold_train))
    weights = total / (num_labels * counts)
    weights = np.sqrt(weights)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=device)


@torch.no_grad()
def inferir_probabilidades_encoder(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
    use_amp: bool = True,
) -> np.ndarray:
    model.eval()
    amp_dtype = torch.bfloat16 if BF16_SUPPORTED else torch.float16
    all_probs = []
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if "token_type_ids" in batch and "modernbert" not in model.config.model_type.lower():
            kwargs["token_type_ids"] = batch["token_type_ids"].to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=use_amp and device.type == "cuda", dtype=amp_dtype):
            outputs = model(**kwargs)
            probs = torch.softmax(outputs.logits.float(), dim=-1).cpu().numpy()
        all_probs.append(probs)
    return normalizar_probabilidades(np.vstack(all_probs))


def treinar_encoder_uma_dobra(
    checkpoint: str,
    textos_tr: List[str],
    y_tr: np.ndarray,
    textos_va: List[str],
    y_va: np.ndarray,
    textos_te: List[str],
    max_length: int = 512,
    num_labels: int = 4,
    epochs: int = 4,
    batch_size: int = 8,
    grad_accum: int = 4,
    lr: float = 2e-5,
    use_grad_checkpointing: bool = False,
    seed: int = SEED,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, Any]]:
    """Realiza Full Fine-Tuning (100% dos parâmetros descongelados) em uma dobra e retorna (val_proba, test_proba, info)."""
    fixar_seeds(seed)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    config = AutoConfig.from_pretrained(checkpoint, num_labels=num_labels)
    if hasattr(config, "reference_compile"):
        config.reference_compile = False

    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, config=config)
    if use_grad_checkpointing and hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    for param in model.parameters():
        param.requires_grad = True

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    model.to(DEVICE)

    collate = criar_collate_fn(tokenizer, max_length=max_length)
    dl_tr = DataLoader(ObrasDocumentDataset(textos_tr, y_tr), batch_size=batch_size, shuffle=True, collate_fn=collate)
    dl_va = DataLoader(ObrasDocumentDataset(textos_va, y_va), batch_size=batch_size * 2, shuffle=False, collate_fn=collate)
    dl_te = DataLoader(ObrasDocumentDataset(textos_te, np.zeros(len(textos_te))), batch_size=batch_size * 2, shuffle=False, collate_fn=collate)

    class_weights = calcular_pesos_classe_tensor(y_tr, num_labels=num_labels, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = max(1, (len(dl_tr) // grad_accum) * epochs)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=max(1, int(0.1 * total_steps)), num_training_steps=total_steps
    )

    amp_dtype = torch.bfloat16 if BF16_SUPPORTED else torch.float16
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda" and not BF16_SUPPORTED))

    best_val_f1 = -1.0
    best_val_proba = None
    best_state_dict = None

    for ep in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(dl_tr, start=1):
            input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
            labels = batch["labels"].to(DEVICE, non_blocking=True)
            kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
            if "token_type_ids" in batch and "modernbert" not in model.config.model_type.lower():
                kwargs["token_type_ids"] = batch["token_type_ids"].to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda"), dtype=amp_dtype):
                outputs = model(**kwargs)
                loss = criterion(outputs.logits, labels) / grad_accum

            if scaler.is_enabled():
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if step % grad_accum == 0 or step == len(dl_tr):
                if scaler.is_enabled():
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        val_proba_ep = inferir_probabilidades_encoder(model, dl_va, DEVICE)
        val_f1_ep = float(f1_score(y_va, np.argmax(val_proba_ep, axis=1), average="macro", zero_division=0))
        if val_f1_ep >= best_val_f1:
            best_val_f1 = val_f1_ep
            best_val_proba = val_proba_ep
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state_dict is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state_dict.items()})
    test_proba = inferir_probabilidades_encoder(model, dl_te, DEVICE)

    info = {
        "total_params": int(total_params),
        "trainable_params": int(trainable_params),
        "percent_trainable": round(100.0 * trainable_params / max(1, total_params), 2),
        "best_val_macro_f1": round(best_val_f1, 4),
    }

    del model, optimizer, scheduler, best_state_dict
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_val_proba, test_proba, info


In [7]:
# ==============================================================================
# SEÇÃO 3.2 — EXECUÇÃO DOS ENCODERS BIDIRECIONAIS (512 TOKENS VS. 2.048 TOKENS)
# ==============================================================================
df_dev_txt = df_dev.loc[mask_texto_dev].reset_index(drop=True)
df_test_txt = df_test.loc[mask_texto_test].reset_index(drop=True)

textos_dev_llm = [formatar_entrada_llm_segura(row, incluir_cabecalho_tabular=True) for _, row in df_dev_txt.iterrows()]
textos_test_llm = [formatar_entrada_llm_segura(row, incluir_cabecalho_tabular=True) for _, row in df_test_txt.iterrows()]
y_dev_txt = df_dev_txt["label_risco"].to_numpy(dtype=np.int64)
y_test_txt = df_test_txt["label_risco"].to_numpy(dtype=np.int64)
folds_dev_txt = df_dev_txt["fold_cv"].to_numpy(dtype=np.int64)

CATALOGO_ENCODERS = [
    {
        "nome": "BERTimbau_FullFT_512",
        "checkpoint": "neuralmind/bert-base-portuguese-cased",
        "max_length": 512,
        "batch_size": 8,
        "grad_accum": 4,
        "lr": 2e-5,
        "grad_ckpt": False,
    },
    {
        "nome": "moBERTo_ModernBERT_PT_2048",
        "checkpoint": "Tropic-AI/moBERTo",
        "max_length": 2048 if EXECUTION_MODE == "FULL_KAGGLE" else 1024,
        "batch_size": 4,
        "grad_accum": 8,
        "lr": 2e-5,
        "grad_ckpt": True,
    },
    {
        "nome": "mmBERT_ModernBERT_Multi_2048",
        "checkpoint": "jhu-clsp/mmBERT-base",
        "max_length": 2048 if EXECUTION_MODE == "FULL_KAGGLE" else 1024,
        "batch_size": 4,
        "grad_accum": 8,
        "lr": 2e-5,
        "grad_ckpt": True,
    },
]

# Registro do baseline histórico local da Sprint 8 (BERTimbau + LoRA r=16, 512 tokens) se disponível
caminho_sprint8_multiclasse = Path("dados_processados/sprint8/predicoes_bertimbau_sprint8.npz")
if not caminho_sprint8_multiclasse.exists():
    caminho_sprint8_multiclasse = Path("../dados_processados/sprint8/predicoes_bertimbau_sprint8.npz")

if caminho_sprint8_multiclasse.exists():
    npz_s8 = np.load(caminho_sprint8_multiclasse, allow_pickle=True)
    proba_s8_test_txt = normalizar_probabilidades(npz_s8["test_probs_multi"])
    proba_s8_oof_txt = normalizar_probabilidades(npz_s8["oof_probs_multi"])
    proba_s8_test_all = PREDICOES_TESTE_INTEGRAL["Hierarquico_C2_LightGBM"].copy()
    proba_s8_test_all[mask_texto_test] = proba_s8_test_txt
    proba_s8_oof_all = PREDICOES_OOF["Hierarquico_C2_LightGBM"].copy()
    proba_s8_oof_all[mask_texto_dev] = proba_s8_oof_txt

    PREDICOES_OOF["BERTimbau_LoRA_r16_512_Sprint8"] = proba_s8_oof_all
    PREDICOES_TESTE_INTEGRAL["BERTimbau_LoRA_r16_512_Sprint8"] = proba_s8_test_all
    PREDICOES_TESTE_TEXTO["BERTimbau_LoRA_r16_512_Sprint8"] = proba_s8_test_txt
    RESULTADOS_MODELOS["BERTimbau_LoRA_r16_512_Sprint8"] = {
        "familia": "2_Encoder_LoRA_Local",
        "espaco_features": "Texto_512_LoRA_r16",
        "tempo_segundos": 0.0,
        "oof_integral": calcular_metricas_completas(y_dev, proba_s8_oof_all),
        "teste_integral_3632": calcular_metricas_completas(y_test, proba_s8_test_all),
        "teste_com_texto_739": calcular_metricas_completas(y_test_txt, proba_s8_test_txt),
    }
    print("[OK] Baseline histórico 'BERTimbau_LoRA_r16_512_Sprint8' carregado para comparação direta.")

for cfg_enc in CATALOGO_ENCODERS:
    nome_enc = cfg_enc["nome"]
    ckpt = cfg_enc["checkpoint"]
    max_len = cfg_enc["max_length"]
    t0 = time.perf_counter()

    if EXECUTION_MODE == "FULL_KAGGLE":
        oof_txt = np.zeros((len(df_dev_txt), 4), dtype=np.float32)
        test_txt_folds = []
        info_fold = {}
        for f in folds_unicos:
            tr_m = folds_dev_txt != f
            va_m = folds_dev_txt == f
            idx_tr = np.where(tr_m)[0]
            idx_va = np.where(va_m)[0]
            val_p, test_p, info_fold = treinar_encoder_uma_dobra(
                checkpoint=ckpt,
                textos_tr=[textos_dev_llm[i] for i in idx_tr],
                y_tr=y_dev_txt[idx_tr],
                textos_va=[textos_dev_llm[i] for i in idx_va],
                y_va=y_dev_txt[idx_va],
                textos_te=textos_test_llm,
                max_length=max_len,
                epochs=4,
                batch_size=cfg_enc["batch_size"],
                grad_accum=cfg_enc["grad_accum"],
                lr=cfg_enc["lr"],
                use_grad_checkpointing=cfg_enc["grad_ckpt"],
                seed=SEED + f,
            )
            oof_txt[va_m] = val_p
            test_txt_folds.append(test_p)
            print(f"    -> [{nome_enc}] Dobra {f}/5 concluída | Val Macro-F1 = {info_fold['best_val_macro_f1']:.4f}")
        test_txt_proba = normalizar_probabilidades(np.mean(test_txt_folds, axis=0))
    else:
        # Modo LOCAL_VALIDATION: Smoke test real na GPU com amostragem estratificada para validar o grafo completo
        rng = np.random.default_rng(SEED)
        idx_tr_sample = []
        idx_va_sample = []
        for c in range(4):
            idx_c_tr = np.where((folds_dev_txt != 1) & (y_dev_txt == c))[0]
            idx_c_va = np.where((folds_dev_txt == 1) & (y_dev_txt == c))[0]
            idx_tr_sample.extend(rng.choice(idx_c_tr, size=min(len(idx_c_tr), 24), replace=False).tolist())
            idx_va_sample.extend(rng.choice(idx_c_va, size=min(len(idx_c_va), 12), replace=False).tolist())

        val_p_smoke, test_p_smoke, info_fold = treinar_encoder_uma_dobra(
            checkpoint=ckpt,
            textos_tr=[textos_dev_llm[i] for i in idx_tr_sample],
            y_tr=y_dev_txt[idx_tr_sample],
            textos_va=[textos_dev_llm[i] for i in idx_va_sample],
            y_va=y_dev_txt[idx_va_sample],
            textos_te=textos_test_llm[:32],
            max_length=min(max_len, 1024),
            epochs=1,
            batch_size=cfg_enc["batch_size"],
            grad_accum=2,
            lr=cfg_enc["lr"],
            use_grad_checkpointing=cfg_enc["grad_ckpt"],
            seed=SEED,
        )
        dt = time.perf_counter() - t0
        print(
            f"  [SMOKE TEST GPU OK] {nome_enc:28s} | Checkpoint={ckpt} | "
            f"Params Treináveis={info_fold['trainable_params']:,d} ({info_fold['percent_trainable']}%) | "
            f"Shape Saída={test_p_smoke.shape} ({dt:.1f}s)"
        )
        continue

    dt = time.perf_counter() - t0
    oof_all = PREDICOES_OOF["Hierarquico_C2_LightGBM"].copy()
    oof_all[mask_texto_dev] = oof_txt
    test_all = PREDICOES_TESTE_INTEGRAL["Hierarquico_C2_LightGBM"].copy()
    test_all[mask_texto_test] = test_txt_proba

    PREDICOES_OOF[nome_enc] = oof_all
    PREDICOES_TESTE_INTEGRAL[nome_enc] = test_all
    PREDICOES_TESTE_TEXTO[nome_enc] = test_txt_proba

    RESULTADOS_MODELOS[nome_enc] = {
        "familia": "2_Encoder_FullFT",
        "espaco_features": f"Texto_{max_len}_FullFT",
        "tempo_segundos": round(dt, 2),
        "info_arquitetura": info_fold,
        "oof_integral": calcular_metricas_completas(y_dev, oof_all),
        "teste_integral_3632": calcular_metricas_completas(y_test, test_all),
        "teste_com_texto_739": calcular_metricas_completas(y_test_txt, test_txt_proba),
    }


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  [SMOKE TEST GPU OK] BERTimbau_FullFT_512         | Checkpoint=neuralmind/bert-base-portuguese-cased | Params Treináveis=108,926,212 (100.0%) | Shape Saída=(32, 4) (6.9s)


W1027 22:31:46.934000 12636 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at Tropic-AI/moBERTo and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  [SMOKE TEST GPU OK] moBERTo_ModernBERT_PT_2048   | Checkpoint=Tropic-AI/moBERTo | Params Treináveis=149,607,940 (100.0%) | Shape Saída=(32, 4) (13.6s)


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at jhu-clsp/mmBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


  [SMOKE TEST GPU OK] mmBERT_ModernBERT_Multi_2048 | Checkpoint=jhu-clsp/mmBERT-base | Params Treináveis=307,533,316 (100.0%) | Shape Saída=(32, 4) (14.3s)


---
## Seção 4 — Família 3: Fine-Tuning de LLMs Decoders em Português com QLoRA 4-bit (`Qwen2.5-3B-Instruct` e `Bode-3.1-8B-Instruct-full`)

Para modelos generativos/decoders de 3B a 8B parâmetros com janelas nativas de 32k a 128k tokens, o *Full Fine-Tuning* em FP16 exigiria > 60 GB de VRAM para pesos, gradientes e estados do AdamW. No Kaggle (16 GB por GPU T4), aplicamos **QLoRA 4-bit NormalFloat (`nf4`)** com *Double Quantization* e adaptadores LoRA de alta capacidade ($r = 64, \alpha = 128$) em todas as projeções lineares de atenção e MLP (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`), reduzindo o uso de VRAM para **~5,2 GB a 7,4 GB**:

1. **`Qwen/Qwen2.5-3B-Instruct`**: Modelo aberto multilíngue de 3B parâmetros de altíssima densidade semântica para textos normativos e financeiros.
2. **`recogna-nlp/Bode-3.1-8B-Instruct-full`**: Modelo de 8B parâmetros baseado no Llama-3.1 e pós-treinado especificamente para o português brasileiro pelo grupo Recogna (UNESP/UFG).

In [8]:
# ==============================================================================
# SEÇÃO 4.1 — PIPELINE QLORA 4-BIT NF4 PARA LLMS DECODERS (QWEN 2.5 & BODE 3.1)
# ==============================================================================
CATALOGO_DECODERS_QLORA = [
    {
        "nome": "Qwen2_5_3B_Instruct_QLoRA_2048",
        "checkpoint": "Qwen/Qwen2.5-3B-Instruct",
        "max_length": 2048,
        "lora_r": 64,
        "lora_alpha": 128,
        "batch_size": 2,
        "grad_accum": 16,
        "lr": 1e-4,
    },
    {
        "nome": "Bode_3_1_8B_Instruct_QLoRA_2048",
        "checkpoint": "recogna-nlp/Bode-3.1-8B-Instruct-full",
        "max_length": 2048,
        "lora_r": 64,
        "lora_alpha": 128,
        "batch_size": 1,
        "grad_accum": 32,
        "lr": 1e-4,
    },
]

def treinar_decoder_qlora_uma_dobra(
    checkpoint: str,
    textos_tr: List[str],
    y_tr: np.ndarray,
    textos_va: List[str],
    y_va: np.ndarray,
    textos_te: List[str],
    max_length: int = 2048,
    lora_r: int = 64,
    lora_alpha: int = 128,
    epochs: int = 3,
    batch_size: int = 2,
    grad_accum: int = 16,
    lr: float = 1e-4,
    seed: int = SEED,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, Any]]:
    """Ajusta um LLM Decoder via QLoRA 4-bit NF4 (r=64, alpha=128) para classificação de sequência."""
    from transformers import BitsAndBytesConfig
    from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

    fixar_seeds(seed)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    compute_dtype = torch.bfloat16 if BF16_SUPPORTED else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )

    base_model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=4,
        quantization_config=bnb_config,
        device_map="auto",
    )
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
    )
    model = get_peft_model(base_model, peft_config)

    collate = criar_collate_fn(tokenizer, max_length=max_length)
    dl_tr = DataLoader(ObrasDocumentDataset(textos_tr, y_tr), batch_size=batch_size, shuffle=True, collate_fn=collate)
    dl_va = DataLoader(ObrasDocumentDataset(textos_va, y_va), batch_size=batch_size * 2, shuffle=False, collate_fn=collate)
    dl_te = DataLoader(ObrasDocumentDataset(textos_te, np.zeros(len(textos_te))), batch_size=batch_size * 2, shuffle=False, collate_fn=collate)

    class_weights = calcular_pesos_classe_tensor(y_tr, num_labels=4, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = max(1, (len(dl_tr) // grad_accum) * epochs)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=max(1, int(0.1 * total_steps)), num_training_steps=total_steps)

    best_val_f1 = -1.0
    best_val_proba = None

    for ep in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(dl_tr, start=1):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=True, dtype=compute_dtype):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits.float(), labels) / grad_accum
            loss.backward()
            if step % grad_accum == 0 or step == len(dl_tr):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        val_proba_ep = inferir_probabilidades_encoder(model, dl_va, DEVICE)
        val_f1_ep = float(f1_score(y_va, np.argmax(val_proba_ep, axis=1), average="macro", zero_division=0))
        if val_f1_ep >= best_val_f1:
            best_val_f1 = val_f1_ep
            best_val_proba = val_proba_ep

    test_proba = inferir_probabilidades_encoder(model, dl_te, DEVICE)
    del model, base_model, optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_val_proba, test_proba, {"best_val_macro_f1": round(best_val_f1, 4)}

if EXECUTION_MODE == "FULL_KAGGLE" and os.environ.get("RUN_DECODERS_QLORA", "1") == "1":
    for cfg_dec in CATALOGO_DECODERS_QLORA:
        nome_dec = cfg_dec["nome"]
        print(f"Iniciando QLoRA 4-bit para {nome_dec} ({cfg_dec['checkpoint']})...")
        oof_txt = np.zeros((len(df_dev_txt), 4), dtype=np.float32)
        test_txt_folds = []
        for f in folds_unicos:
            tr_m = folds_dev_txt != f
            va_m = folds_dev_txt == f
            idx_tr = np.where(tr_m)[0]
            idx_va = np.where(va_m)[0]
            val_p, test_p, info_d = treinar_decoder_qlora_uma_dobra(
                checkpoint=cfg_dec["checkpoint"],
                textos_tr=[textos_dev_llm[i] for i in idx_tr],
                y_tr=y_dev_txt[idx_tr],
                textos_va=[textos_dev_llm[i] for i in idx_va],
                y_va=y_dev_txt[idx_va],
                textos_te=textos_test_llm,
                max_length=cfg_dec["max_length"],
                lora_r=cfg_dec["lora_r"],
                lora_alpha=cfg_dec["lora_alpha"],
                epochs=3,
                batch_size=cfg_dec["batch_size"],
                grad_accum=cfg_dec["grad_accum"],
                lr=cfg_dec["lr"],
                seed=SEED + f,
            )
            oof_txt[va_m] = val_p
            test_txt_folds.append(test_p)
        test_txt_proba = normalizar_probabilidades(np.mean(test_txt_folds, axis=0))
        oof_all = PREDICOES_OOF["Hierarquico_C2_LightGBM"].copy()
        oof_all[mask_texto_dev] = oof_txt
        test_all = PREDICOES_TESTE_INTEGRAL["Hierarquico_C2_LightGBM"].copy()
        test_all[mask_texto_test] = test_txt_proba
        PREDICOES_OOF[nome_dec] = oof_all
        PREDICOES_TESTE_INTEGRAL[nome_dec] = test_all
        PREDICOES_TESTE_TEXTO[nome_dec] = test_txt_proba
        RESULTADOS_MODELOS[nome_dec] = {
            "familia": "3_Decoder_QLoRA_4bit",
            "espaco_features": f"Texto_{cfg_dec['max_length']}_QLoRA_r64",
            "oof_integral": calcular_metricas_completas(y_dev, oof_all),
            "teste_integral_3632": calcular_metricas_completas(y_test, test_all),
            "teste_com_texto_739": calcular_metricas_completas(y_test_txt, test_txt_proba),
        }
else:
    print("[INFO] Pipeline QLoRA 4-bit (Qwen 2.5 3B & Bode 3.1 8B) pronto e configurado para execução em GPU Kaggle (EXECUTION_MODE='FULL_KAGGLE').")


[INFO] Pipeline QLoRA 4-bit (Qwen 2.5 3B & Bode 3.1 8B) pronto e configurado para execução em GPU Kaggle (EXECUTION_MODE='FULL_KAGGLE').


---
## Seção 5 — Fusão Híbrida Calibrada em OOF, Inferência Estatística (Bootstrap $B = 1.000$) e Explicabilidade (XAI)

Nesta seção final:
1. **Calibração Sem Vazamento (*Out-Of-Fold Late Fusion*)**:
   - Otimizamos os pesos convexos $\alpha \in [0, 1]$ entre os melhores modelos multimodais (`Hierarquico_C2_LightGBM`, `Hierarquico_C2_CatBoost`, `Hierarquico_C2_XGBoost` e os Encoders de Longo Contexto) **exclusivamente sobre as predições OOF de `treino_dev`**, avaliando o ensemble resultante uma única vez no `teste_retido`.
2. **Intervalos de Confiança via Bootstrap ($B = 1.000$) e Testes Pareados**:
   - Calculamos IC 95% para `Macro-F1`, `Balanced Accuracy` e `MCC`, além do $p$-value pareado bilateral contra o campeão de referência `Hierarquico_C2_LightGBM`.
3. **Explicabilidade Auditável (`TreeSHAP`)**:
   - Extraímos as importâncias globais SHAP do Estágio 1 (Detecção de Risco) e do Estágio 2 (Atribuição Causal), quantificando a contribuição relativa das variáveis tabulares `C1` versus os *embeddings* textuais `gte_c2`.

In [9]:
# ==============================================================================
# SEÇÃO 5.1 — CALIBRAÇÃO DO ENSEMBLE EM OOF, BOOTSTRAP (B=1.000), SHAP E EXPORTAÇÃO
# ==============================================================================
# 1. Calibração de Ensemble Multimodal em OOF (Hierarquico_C2_LightGBM + Hierarquico_C2_CatBoost + Hierarquico_C2_XGBoost)
oof_lgb = PREDICOES_OOF["Hierarquico_C2_LightGBM"]
oof_cat = PREDICOES_OOF["Hierarquico_C2_CatBoost"]
oof_xgb = PREDICOES_OOF["Hierarquico_C2_XGBoost"]

melhor_f1_oof = -1.0
melhores_pesos = (1.0, 0.0, 0.0)

for w_lgb in np.linspace(0.2, 1.0, 9):
    for w_cat in np.linspace(0.0, 1.0 - w_lgb, 9):
        w_xgb = max(0.0, round(1.0 - w_lgb - w_cat, 4))
        oof_blend = normalizar_probabilidades(w_lgb * oof_lgb + w_cat * oof_cat + w_xgb * oof_xgb)
        f1_val = float(f1_score(y_dev, np.argmax(oof_blend, axis=1), average="macro", zero_division=0))
        if f1_val > melhor_f1_oof:
            melhor_f1_oof = f1_val
            melhores_pesos = (round(float(w_lgb), 3), round(float(w_cat), 3), round(float(w_xgb), 3))

w_lgb, w_cat, w_xgb = melhores_pesos
oof_ens_arvores = normalizar_probabilidades(w_lgb * oof_lgb + w_cat * oof_cat + w_xgb * oof_xgb)
test_ens_arvores = normalizar_probabilidades(
    w_lgb * PREDICOES_TESTE_INTEGRAL["Hierarquico_C2_LightGBM"]
    + w_cat * PREDICOES_TESTE_INTEGRAL["Hierarquico_C2_CatBoost"]
    + w_xgb * PREDICOES_TESTE_INTEGRAL["Hierarquico_C2_XGBoost"]
)

PREDICOES_OOF["Ensemble_Hierarquico_C2_LGB_CAT_XGB"] = oof_ens_arvores
PREDICOES_TESTE_INTEGRAL["Ensemble_Hierarquico_C2_LGB_CAT_XGB"] = test_ens_arvores
PREDICOES_TESTE_TEXTO["Ensemble_Hierarquico_C2_LGB_CAT_XGB"] = test_ens_arvores[mask_texto_test]
RESULTADOS_MODELOS["Ensemble_Hierarquico_C2_LGB_CAT_XGB"] = {
    "familia": "4_Ensemble_Multimodal_OOF",
    "espaco_features": f"C2_Blend(LGB={w_lgb},CAT={w_cat},XGB={w_xgb})",
    "tempo_segundos": 0.5,
    "oof_integral": calcular_metricas_completas(y_dev, oof_ens_arvores),
    "teste_integral_3632": calcular_metricas_completas(y_test, test_ens_arvores),
    "teste_com_texto_739": calcular_metricas_completas(y_test_txt, test_ens_arvores[mask_texto_test]),
}

# Se houver encoder de longo contexto ou BERTimbau treinado, calibrar Fusão Híbrida (Árvore C2 + LLM) nas obras com texto
candidatos_llm = [
    k for k in PREDICOES_TESTE_TEXTO.keys()
    if any(p in k for p in ["moBERTo", "mmBERT", "BERTimbau", "Qwen", "Bode"])
]
for nome_llm in candidatos_llm:
    oof_llm_txt = PREDICOES_OOF[nome_llm][mask_texto_dev]
    oof_c2_txt = oof_ens_arvores[mask_texto_dev]
    best_alpha = 0.0
    best_f1_hib = -1.0
    for alpha in np.linspace(0.0, 0.6, 25):
        blend_txt = normalizar_probabilidades((1.0 - alpha) * oof_c2_txt + alpha * oof_llm_txt)
        f1_h = float(f1_score(y_dev_txt, np.argmax(blend_txt, axis=1), average="macro", zero_division=0))
        if f1_h > best_f1_hib:
            best_f1_hib = f1_h
            best_alpha = round(float(alpha), 3)

    oof_hib = oof_ens_arvores.copy()
    oof_hib[mask_texto_dev] = normalizar_probabilidades(
        (1.0 - best_alpha) * oof_c2_txt + best_alpha * oof_llm_txt
    )
    test_hib = test_ens_arvores.copy()
    test_hib[mask_texto_test] = normalizar_probabilidades(
        (1.0 - best_alpha) * test_ens_arvores[mask_texto_test] + best_alpha * PREDICOES_TESTE_TEXTO[nome_llm]
    )
    nome_hibrido = f"Hibrido_OOF_C2_Mais_{nome_llm}"
    PREDICOES_OOF[nome_hibrido] = oof_hib
    PREDICOES_TESTE_INTEGRAL[nome_hibrido] = test_hib
    PREDICOES_TESTE_TEXTO[nome_hibrido] = test_hib[mask_texto_test]
    RESULTADOS_MODELOS[nome_hibrido] = {
        "familia": "4_Hibrido_Arvore_LLM",
        "espaco_features": f"LateFusion(alpha_llm={best_alpha})",
        "tempo_segundos": 0.2,
        "oof_integral": calcular_metricas_completas(y_dev, oof_hib),
        "teste_integral_3632": calcular_metricas_completas(y_test, test_hib),
        "teste_com_texto_739": calcular_metricas_completas(y_test_txt, test_hib[mask_texto_test]),
    }

# 2. Inferência Estatística: Bootstrap (B=1.000 em FULL_KAGGLE / B=300 em LOCAL_VALIDATION) e p-values pareados
N_BOOT = 1000 if EXECUTION_MODE == "FULL_KAGGLE" else 300
rng_boot = np.random.default_rng(SEED)
n_test = len(y_test)
boot_indices = rng_boot.integers(0, n_test, size=(N_BOOT, n_test))

pred_ref = np.argmax(PREDICOES_TESTE_INTEGRAL["Hierarquico_C2_LightGBM"], axis=1)
f1_ref_boot = np.array([
    f1_score(y_test[idx], pred_ref[idx], average="macro", zero_division=0)
    for idx in boot_indices
])

tabela_final_linhas = []
for nome_mod, res in RESULTADOS_MODELOS.items():
    pred_mod = np.argmax(PREDICOES_TESTE_INTEGRAL[nome_mod], axis=1)
    f1_mod_boot = np.array([
        f1_score(y_test[idx], pred_mod[idx], average="macro", zero_division=0)
        for idx in boot_indices
    ])
    ic_low, ic_high = np.percentile(f1_mod_boot, [2.5, 97.5])
    diffs = f1_mod_boot - f1_ref_boot
    if nome_mod == "Hierarquico_C2_LightGBM":
        p_val = 1.0
    else:
        p_one = min(float(np.mean(diffs <= 0.0)), float(np.mean(diffs >= 0.0)))
        p_val = min(1.0, 2.0 * p_one)

    res["estatistica_bootstrap_teste_3632"] = {
        "n_bootstrap": N_BOOT,
        "macro_f1_ic95": [round(float(ic_low), 4), round(float(ic_high), 4)],
        "delta_f1_vs_Hierarquico_C2_LightGBM": round(float(diffs.mean()), 4),
        "p_value_pareado_bilateral": round(float(p_val), 4),
    }

    t_all = res["teste_integral_3632"]
    t_txt = res["teste_com_texto_739"]
    tabela_final_linhas.append({
        "Modelo": nome_mod,
        "Família": res["familia"],
        "OOF Macro-F1": res["oof_integral"]["macro_f1"],
        "Teste(3632) Macro-F1": t_all["macro_f1"],
        "IC 95% Macro-F1": f"[{ic_low:.4f}, {ic_high:.4f}]",
        "Teste(3632) Bal.Acc": t_all["balanced_accuracy"],
        "Teste(3632) MCC": t_all["mcc"],
        "Sens. Risco (Bin)": t_all["bin_sensibilidade_risco"],
        "Espec. Regular (Bin)": t_all["bin_especificidade_regular"],
        "TesteTexto(739) Macro-F1": t_txt["macro_f1"],
        "TesteTexto(739) MCC": t_txt["mcc"],
        "p-value vs C2_LGBM": round(float(p_val), 4),
    })

df_tabela_final = pd.DataFrame(tabela_final_linhas).sort_values("Teste(3632) Macro-F1", ascending=False)
print(f"\n[TABELA CONSOLIDADA FINAL — AVALIAÇÃO NO TESTE RETIDO CONGELADO (BOOTSTRAP B={N_BOOT})]")
display(df_tabela_final)

# 3. Explicabilidade TreeSHAP Nativa (Lundberg & Lee, C++ LightGBM `pred_contrib=True`) no Campeão Hierarquico_C2_LightGBM
modelo_campeao = MODELOS_TREINADOS_ARVORE["Hierarquico_C2_LightGBM"]
amostra_shap_idx = np.where(mask_texto_test)[0][:200]
X_shap = X_comb_test[amostra_shap_idx]

# Estágio 1: Binário (retorna N x (D + 1), última coluna é o bias/expected_value)
contribs_s1 = np.asarray(modelo_campeao.stage1.predict(X_shap, pred_contrib=True), dtype=np.float32)
shap_vals_s1 = contribs_s1[:, :-1]
imp_abs_s1 = np.abs(shap_vals_s1).mean(axis=0)

soma_tab_s1 = float(imp_abs_s1[:6].sum())
soma_emb_s1 = float(imp_abs_s1[6:].sum())
total_s1 = max(1e-9, soma_tab_s1 + soma_emb_s1)

# Estágio 2: Multiclasse Causal (3 classes x (D + 1))
contribs_s2 = np.asarray(modelo_campeao.stage2.predict(X_shap, pred_contrib=True), dtype=np.float32)
contribs_s2 = contribs_s2.reshape(len(X_shap), 3, X_shap.shape[1] + 1)
shap_vals_s2 = contribs_s2[:, :, :-1]
imp_abs_s2 = np.abs(shap_vals_s2).mean(axis=(0, 1))
soma_tab_s2 = float(imp_abs_s2[:6].sum())
soma_emb_s2 = float(imp_abs_s2[6:].sum())
total_s2 = max(1e-9, soma_tab_s2 + soma_emb_s2)

resumo_xai_shap = {
    "estagio1_participacao_tabular_c1_pct": round(100.0 * soma_tab_s1 / total_s1, 2),
    "estagio1_participacao_embeddings_gte_c2_pct": round(100.0 * soma_emb_s1 / total_s1, 2),
    "estagio2_participacao_tabular_c1_pct": round(100.0 * soma_tab_s2 / total_s2, 2),
    "estagio2_participacao_embeddings_gte_c2_pct": round(100.0 * soma_emb_s2 / total_s2, 2),
    "importancia_atributos_tabulares_c1_estagio1": {
        COLUNAS_TABULARES_C1[i]: round(float(imp_abs_s1[i]), 5) for i in range(6)
    },
    "importancia_atributos_tabulares_c1_estagio2": {
        COLUNAS_TABULARES_C1[i]: round(float(imp_abs_s2[i]), 5) for i in range(6)
    },
}
print("\n[EXPLICABILIDADE TREESHAP — PARTICIPAÇÃO MODAL NOS ESTÁGIOS 1 E 2]")
print(json.dumps(resumo_xai_shap, indent=2, ensure_ascii=False))

# 4. Exportação dos Artefatos Finais (JSON e CSV)
caminho_csv = OUTPUT_DIR / "tabela_comparativa_tcc.csv"
caminho_json = OUTPUT_DIR / "resultados_kaggle_consolidados.json"

df_tabela_final.to_csv(caminho_csv, index=False, encoding="utf-8-sig")
relatorio_consolidado = {
    "versao_pipeline": "2.0_auditado",
    "modo_execucao": EXECUTION_MODE,
    "diagnostico_janela_tokens": relatorio_tokens,
    "pesos_ensemble_c2_oof": {"lightgbm": w_lgb, "catboost": w_cat, "xgboost": w_xgb},
    "explicabilidade_treeshap": resumo_xai_shap,
    "modelos": RESULTADOS_MODELOS,
}
with open(caminho_json, "w", encoding="utf-8") as f:
    json.dump(relatorio_consolidado, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 80)
print(f"ARTEFATOS EXPORTADOS COM SUCESSO:")
print(f"  1) CSV : {caminho_csv}")
print(f"  2) JSON: {caminho_json}")
print("=" * 80)



[TABELA CONSOLIDADA FINAL — AVALIAÇÃO NO TESTE RETIDO CONGELADO (BOOTSTRAP B=300)]


,Modelo,Família,OOF Macro-F1,Teste(3632) Macro-F1,IC 95% Macro-F1,Teste(3632) Bal.Acc,Teste(3632) MCC,Sens. Risco (Bin),Espec. Regular (Bin),TesteTexto(739) Macro-F1,TesteTexto(739) MCC,p-value vs C2_LGBM
8,CatBoost_C2_Multimodal,1_Arvore_Flat,0.5431,0.5932,"[0.5360, 0.6387]",0.6966,0.5817,0.8510,0.8985,0.6137,0.6162,0.0067
13,Ensemble_Hierarquico_C2_LGB_CAT_XGB,4_Ensemble_Multimodal_OOF,0.5626,0.5405,"[0.5048, 0.5798]",0.6399,0.5692,0.8377,0.9081,0.6120,0.5966,0.9600
10,Hierarquico_C2_LightGBM,1_Arvore_Hierarquica,0.5384,0.5377,"[0.5038, 0.5693]",0.6945,0.5508,0.9040,0.8814,0.5810,0.5905,1.0000
11,Hierarquico_C2_XGBoost,1_Arvore_Hierarquica,0.5424,0.5361,"[0.4993, 0.5684]",0.6938,0.5510,0.9040,0.8808,0.5888,0.5922,0.7933
12,Hierarquico_C2_CatBoost,1_Arvore_Hierarquica,0.5545,0.5322,"[0.5062, 0.5550]",0.5897,0.5568,0.7219,0.9369,0.6097,0.5693,0.7400
4,LightGBM_C2_Multimodal,1_Arvore_Flat,0.5155,0.5166,"[0.4892, 0.5428]",0.7185,0.5258,0.9272,0.8565,0.6119,0.6036,0.0467
6,XGBoost_C2_Multimodal,1_Arvore_Flat,0.5146,0.5121,"[0.4868, 0.5360]",0.7302,0.5233,0.9503,0.8405,0.6089,0.6120,0.0533
7,CatBoost_C1_Tabular,1_Arvore_Flat,0.4868,0.5054,"[0.4528, 0.5541]",0.6128,0.4193,0.7185,0.8652,0.5982,0.4713,0.0667
2,RandomForest_C2_Multimodal,1_Arvore_Flat,0.4909,0.4775,"[0.4473, 0.5042]",0.6594,0.4891,0.9106,0.8186,0.5047,0.4801,0.0000
1,RandomForest_C1_Tabular,1_Arvore_Flat,0.4542,0.4616,"[0.4137, 0.5073]",0.6448,0.4029,0.8675,0.7745,0.5972,0.5797,0.0000



[EXPLICABILIDADE TREESHAP — PARTICIPAÇÃO MODAL NOS ESTÁGIOS 1 E 2]
{
  "estagio1_participacao_tabular_c1_pct": 32.94,
  "estagio1_participacao_embeddings_gte_c2_pct": 67.06,
  "estagio2_participacao_tabular_c1_pct": 65.3,
  "estagio2_participacao_embeddings_gte_c2_pct": 34.7,
  "importancia_atributos_tabulares_c1_estagio1": {
    "valor_investimento": 0.0974,
    "valor_desbloqueado": 0.16984,
    "exec_fisica": 1.71074,
    "exec_financeira": 0.13883,
    "base_cat": 0.38805,
    "uf_cat": 0.03963
  },
  "importancia_atributos_tabulares_c1_estagio2": {
    "valor_investimento": 0.23343,
    "valor_desbloqueado": 0.10584,
    "exec_fisica": 0.1334,
    "exec_financeira": 0.06734,
    "base_cat": 1.26049,
    "uf_cat": 0.05375
  }
}

ARTEFATOS EXPORTADOS COM SUCESSO:
  1) CSV : C:\Users\lucas\OneDrive\Documentos\TCC\resultados_validacao_local\tabela_comparativa_tcc.csv
  2) JSON: C:\Users\lucas\OneDrive\Documentos\TCC\resultados_validacao_local\resultados_kaggle_consolidados.json
